In [1]:
# để import ở đây hết nhé mọi người 

import os
import glob
import pandas as pd
import numpy as np

In [2]:
 # cái này để install kaggle hub giúp tải data từ kaggle

!pip install kagglehub pandas

In [3]:
# ---- Option B: set this to your CSV file to skip the download (leave None to use Option A) ----
MANUAL_CSV_PATH = None     # e.g. r"C:\Users\you\Downloads\GOOGL.csv"

DATASET = "varpit94/google-stock-data"  # link của cái chỗ tải trên kaggle

if MANUAL_CSV_PATH is None:   # chỉ chạy khi không có file csv, nếu có thì bỏ qua bước download   
    import kagglehub    
    # kagglehub.login()    # uncomment if it asks you to authenticate
    dataset_dir = kagglehub.dataset_download(DATASET)
    print("Downloaded to:", dataset_dir)
else:
    dataset_dir = os.path.dirname(MANUAL_CSV_PATH)
    print("Using manual path folder:", dataset_dir)

c:\Users\dummyking\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloaded to: C:\Users\dummyking\.cache\kagglehub\datasets\varpit94\google-stock-data\versions\6


In [4]:
if MANUAL_CSV_PATH is not None:
    csv_files = [MANUAL_CSV_PATH]
else:
    csv_files = sorted(glob.glob(os.path.join(dataset_dir, "**", "*.csv"), recursive=True))

if not csv_files:
    raise FileNotFoundError(f"No CSV found in {dataset_dir}. Download it manually and set MANUAL_CSV_PATH.")

for i, f in enumerate(csv_files):
    print(i, "->", f, f"({os.path.getsize(f)/1024:.1f} KB)")

0 -> C:\Users\dummyking\.cache\kagglehub\datasets\varpit94\google-stock-data\versions\6\GOOGL.csv (325.7 KB)


In [5]:
# cái này sẽ đọc csv và trả về dataframe nek 

CSV_PATH = csv_files[0]
print("Reading:", CSV_PATH)

df = pd.read_csv(CSV_PATH)
print("Shape (rows, columns):", df.shape)
df.head()

Reading: C:\Users\dummyking\.cache\kagglehub\datasets\varpit94\google-stock-data\versions\6\GOOGL.csv
Shape (rows, columns): (4431, 7)


,Date,Open,High,Low,Close,Adj Close,Volume
0,2004-08-19,50.050049,52.082081,48.028027,50.220219,50.220219,44659096
1,2004-08-20,50.555557,54.594597,50.300301,54.209209,54.209209,22834343
2,2004-08-23,55.430431,56.796799,54.579578,54.754753,54.754753,18256126
3,2004-08-24,55.675674,55.855858,51.836838,52.487488,52.487488,15247337
4,2004-08-25,52.532532,54.054054,51.991993,53.053055,53.053055,9188602


In [6]:
# kiểm tra cấu trúc của files

print("Columns:", list(df.columns))
print()
df.info()

Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

<class 'pandas.DataFrame'>
RangeIndex: 4431 entries, 0 to 4430
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       4431 non-null   str    
 1   Open       4431 non-null   float64
 2   High       4431 non-null   float64
 3   Low        4431 non-null   float64
 4   Close      4431 non-null   float64
 5   Adj Close  4431 non-null   float64
 6   Volume     4431 non-null   int64  
dtypes: float64(5), int64(1), str(1)
memory usage: 285.7 KB


In [7]:
print("Summary statistics:")
display(df.describe())  # dùng chức năng describe để xem các thông số

Summary statistics:


,Open,High,Low,Close,Adj Close,Volume
count,4431.000000,4431.000000,4431.000000,4431.000000,4431.000000,4.431000e+03
mean,693.087345,699.735595,686.078751,693.097367,693.097367,6.444992e+06
std,645.118799,651.331215,638.579488,645.187806,645.187806,7.690351e+06
min,49.644646,50.920921,48.028027,50.055054,50.055054,4.656000e+05
25%,248.558563,250.853355,245.813309,248.415916,248.415916,1.695600e+06
50%,434.924927,437.887878,432.687683,435.330322,435.330322,3.778418e+06
75%,1007.364990,1020.649994,997.274994,1007.790008,1007.790008,8.002390e+06
max,3025.000000,3030.929932,2977.979980,2996.770020,2996.770020,8.215117e+07


In [8]:
# kiểm tra dữ liệu có bị thiếu hay không, có bị trùng lặp hay không, và nếu có cột Date thì in ra ngày đầu tiên và ngày cuối cùng

print("Missing values per column:")
display(df.isna().sum())

print("Duplicate rows:", df.duplicated().sum())
if "Date" in df.columns:
    print("First date in file:", df["Date"].iloc[0])
    print("Last date in file :", df["Date"].iloc[-1])

Missing values per column:


Date         0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64

Duplicate rows: 0
First date in file: 2004-08-19
Last date in file : 2022-03-24


In [9]:
# Tạo cửa sổ 30 ngày liên tiếp

WINDOW = 30  # dùng 30 ngày liên tiếp để dự đoán ngày kế tiếp

# Sort theo ngày tăng dần
df["Date"] = pd.to_datetime(df["Date"])
df_sorted = df.sort_values("Date").reset_index(drop=True)

# Chọn feature, lấy ra mảng số
FEATURES = ["Open", "High", "Low", "Close", "Volume"]   # bỏ Adj Close vì giống hệt Close
data = df_sorted[FEATURES].values.astype(np.float32)    # shape (số_ngày, số_feature)
TARGET_COL = FEATURES.index("Close")                    # cột cần dự đoán


# Hàm tạo cửa sổ
def make_windows(data, target_col, window=WINDOW):
    """
    data       : mảng 2D (số_ngày, số_feature), đã sort theo ngày
    target_col : chỉ số cột cần dự đoán (Close)
    return     : X (số_sample, window, số_feature), y (số_sample, 1)
    """
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i : i + window])           # 30 ngày liên tiếp: i .. i+29
        y.append(data[i + window, target_col])   # Close của ngày kế tiếp: i+30
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32).reshape(-1, 1)
    return X, y


X, y = make_windows(data, TARGET_COL)

# Kiểm tra input và dimension
print("data shape:", data.shape)
print("X shape:", X.shape, "-> (số sample, 30 ngày, số feature)")
print("y shape:", y.shape, "-> (số sample, 1)")

assert X.shape == (len(data) - WINDOW, WINDOW, data.shape[1])
assert y.shape == (len(data) - WINDOW, 1)
assert np.isfinite(X).all() and np.isfinite(y).all()      # không có NaN / inf
assert np.array_equal(X[0], data[0:30])                   # sample đầu: ngày 0..29
assert y[0, 0] == data[30, TARGET_COL]                    # nhãn = Close ngày 30
assert np.array_equal(X[1][:-1], X[0][1:])                # cửa sổ trượt đúng 1 ngày
assert y[-1, 0] == data[-1, TARGET_COL]                   # sample cuối dùng ngày cuối
print("Tất cả kiểm tra đều OK")

data shape: (4431, 5)
X shape: (4401, 30, 5) -> (số sample, 30 ngày, số feature)
y shape: (4401, 1) -> (số sample, 1)
Tất cả kiểm tra đều OK
